# Geography Chatbot with LangGraph

A short, interactive geography chatbot for students. Add `OPENAI_API_KEY` to Colab Secrets, enable notebook access, then run the cells from top to bottom.

In [ ]:
%pip -q install -U langgraph langchain langchain-openai


In [ ]:
import os
from typing import Annotated, TypedDict

from google.colab import userdata
from langchain.chat_models import init_chat_model
from langchain_core.messages import AnyMessage, HumanMessage, SystemMessage
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
model = init_chat_model("openai:gpt-6-luna")

In [ ]:
class ChatState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

SYSTEM = SystemMessage(
    content=(
        "You are a friendly chatbot for students. Answer questions about geography only. "
        "If asked about another topic, politely explain that you can help only with geography."
    )
)

def chatbot(state: ChatState):
    reply = model.invoke([SYSTEM] + state["messages"])
    return {"messages": [reply]}

builder = StateGraph(ChatState)
builder.add_node("chatbot", chatbot)
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)
graph = builder.compile(checkpointer=InMemorySaver())

config = {"configurable": {"thread_id": "1"}}

In [ ]:
while True:
    text = input("You (type 'stop' to end): ").strip()
    if text.lower() == "stop":
        print("Goodbye!")
        break

    result = graph.invoke(
        {"messages": [HumanMessage(text)]},
        config,
    )
    print("Bot:", result["messages"][-1].content)